# LLM Evaluation — From Metrics to Custom Eval Pipelines

**Session 7 — Applied Natural Language Processing**

Good prompting is only half the picture. How do we know if our prompts actually work?

This notebook covers three tiers of LLM evaluation:
- **Tier A — Classical metrics**: BLEU, ROUGE, BERTScore, perplexity
- **Tier B — Eval tools**: Promptfoo for prompt testing
- **Tier C — Custom pipelines**: DIY eval loop and LLM-as-judge

**Part 1** requires no API key. **Part 2** requires a GitHub token (set `GITHUB_TOKEN` env var).

In [ ]:
# Part 1 dependencies — no API key needed
# Uncomment to install if not already in your environment:
# !pip install evaluate bert-score transformers torch

import warnings
warnings.filterwarnings("ignore")

import evaluate
from bert_score import score as bert_score
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

print("Part 1 dependencies ready.")

In [ ]:
# Part 2 dependencies — requires GITHUB_TOKEN
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ.get("GITHUB_TOKEN", "set-your-github-token"),
)
MODEL = "gpt-4o-mini"

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print("Part 2 dependencies ready.")

---

## Part 1 — Classical Metrics (No API Key Required)

Classical NLP metrics predate LLMs but remain widely used. They measure surface-level or semantic similarity between a model's output and a reference (gold-standard) text.

We'll use three summarisation examples to compare the metrics:
- **Good summary**: captures key points, well-phrased
- **Bad summary**: misses key points, poor quality
- **Synonym summary**: correct meaning, different words (tests metric sensitivity to synonyms)

### 1a — BLEU and ROUGE

**BLEU** (Bilingual Evaluation Understudy) measures n-gram precision — how many n-grams in the model output appear in the reference. Designed for machine translation.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram recall — how many reference n-grams appear in the model output. Designed for summarisation.

Both suffer from the same limitation: they measure surface overlap, not meaning. A synonym earns zero credit.

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# Source article (what we're summarising)
source = (
    "The city council voted unanimously to approve a new public transport initiative. "
    "The plan includes expanding the metro network by 12 new stations, introducing "
    "electric buses on all major routes, and building 50 new cycling lanes. "
    "The project is expected to reduce car usage by 30% over five years."
)

# Reference: what a human would write as the ideal summary
reference = "The council approved a transport plan to expand metro, add electric buses, and build cycling lanes, aiming to cut car use by 30%."

# Three candidate summaries
candidates = {
    "Good summary":    "The city council approved a transport initiative to expand the metro by 12 stations, introduce electric buses, and add cycling lanes, targeting a 30% reduction in car use.",
    "Bad summary":     "The council had a meeting and discussed various city matters including transport and other infrastructure topics.",
    "Synonym summary": "The municipal council unanimously endorsed a transit scheme to broaden the subway network, deploy electric coaches, and create bicycle paths, aiming to decrease vehicle traffic by 30%.",
}

print(f"{'Summary':<20} {'BLEU':>8} {'ROUGE-1':>10} {'ROUGE-L':>10}")
print("-" * 52)

for name, candidate in candidates.items():
    bleu = bleu_metric.compute(predictions=[candidate], references=[[reference]])
    rouge = rouge_metric.compute(predictions=[candidate], references=[reference])
    print(f"{name:<20} {bleu['bleu']:>8.3f} {rouge['rouge1']:>10.3f} {rouge['rougeL']:>10.3f}")

**What to notice:**
- The **good summary** scores well — it uses similar words to the reference.
- The **synonym summary** may score poorly on BLEU despite being semantically correct — it uses different words.
- The **bad summary** scores near zero, as expected.
- This illustrates the core limitation: BLEU and ROUGE reward word overlap, not meaning.

---

### 1b — BERTScore

BERTScore (Zhang et al., 2020) addresses the synonym problem. Instead of counting matching words, it uses contextual embeddings from BERT to compute semantic similarity. A paraphrase scores highly even if it shares no words with the reference.

In [ ]:
# Run BERTScore on the same candidates
references_list = [reference] * len(candidates)
candidates_list = list(candidates.values())
candidate_names = list(candidates.keys())

# Compute BERTScore
P, R, F1 = bert_score(candidates_list, references_list, lang="en", verbose=False)

print(f"{'Summary':<20} {'BERTScore F1':>14}")
print("-" * 36)
for name, f1 in zip(candidate_names, F1):
    print(f"{name:<20} {f1.item():>14.3f}")

**What to notice:**
- The **synonym summary** should now score similarly to (or better than) the good summary — BERTScore rewards semantic equivalence.
- The **bad summary** still scores low, even with BERTScore.
- This shows the value of embedding-based metrics for tasks where paraphrasing is acceptable (e.g., summarisation, translation).

---

### 1c — Perplexity with GPT-2

**Perplexity** measures how surprised a language model is by a piece of text. Lower perplexity = the model finds the text more probable = the text is more fluent.

Perplexity does **not** measure factual accuracy — a fluent lie scores better than an awkward truth.

We use GPT-2 (a small, open model) as the language model for this demo.

In [ ]:
# Load GPT-2 for perplexity computation
print("Loading GPT-2...")
model_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
gpt2_model = GPT2LMHeadModel.from_pretrained(model_name)
gpt2_model.eval()
print("GPT-2 loaded.")

def compute_perplexity(text):
    """Compute perplexity of a text using GPT-2."""
    encodings = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = gpt2_model(**encodings, labels=encodings["input_ids"])
    loss = outputs.loss
    return torch.exp(loss).item()

# Compare fluent vs disfluent text
examples = {
    "Fluent": "The restaurant was beautifully decorated and the service was excellent.",
    "Disfluent": "Restaurant the was decorated beautifully excellent service the was and.",
    "Factually wrong but fluent": "The Eiffel Tower is located in London, where it was built in 1889.",
}

print(f"\n{'Text type':<35} {'Perplexity':>12}")
print("-" * 49)
for label, text in examples.items():
    ppl = compute_perplexity(text)
    print(f"{label:<35} {ppl:>12.1f}")

**What to notice:**
- The disfluent sentence has much higher perplexity — GPT-2 finds it improbable.
- The factually wrong but fluent sentence has **low perplexity** — it reads naturally.
- This demonstrates why perplexity alone cannot catch hallucinations.
- Use perplexity to compare fluency across models, not to verify factual accuracy.

---

### 1d — Leaderboard Exploration

No code needed — this section is guided reading.

**Key leaderboards to explore:**

| Leaderboard | What it measures | URL |
|---|---|---|
| HF Open LLM Leaderboard | Open-source model benchmarks (MMLU, HellaSwag, etc.) | [huggingface.co/spaces/open-llm-leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) |
| LMSYS Chatbot Arena | Human preference (Elo rating from real user votes) | [arena.ai](https://arena.ai) |
| Artificial Analysis | Intelligence + speed + cost + latency | [artificialanalysis.ai](https://artificialanalysis.ai) |

**Questions to ask when reading a leaderboard:**
1. Is the benchmark saturated? (If top models all score 90%+, small differences are noise.)
2. Is this a reasoning model being compared on reasoning benchmarks only?
3. What is *not* reported? (Labs choose which benchmarks to publish.)
4. Does strong benchmark performance predict strong performance on *my* task? (Usually not directly.)

**Key insight:** Leaderboards give you an evidence-based shortlist. They replace uninformed guessing — but they're proxies for your actual use case. The further your task is from standard benchmarks, the more critical your own evaluation becomes.

---

## Part 2 — Build Your Own Eval Pipeline

*(Requires GitHub Token — set `GITHUB_TOKEN` in your environment)*

Classical metrics measure output quality in isolation. But for real projects, you need evaluation tailored to your specific task. This section builds a custom eval pipeline from first principles, then shows how tools like promptfoo make this scalable.

### 2a — Promptfoo

[Promptfoo](https://github.com/promptfoo/promptfoo) is an open-source CLI tool for prompt testing and model comparison. You define test cases in YAML, run them against one or more models, and get a visual comparison report.

It's particularly useful for:
- Comparing the same prompt across different models
- Regression testing when you update a prompt
- AT2-style projects where you need systematic prompt evaluation

**A working example is in this session's `getting-started/` folder** — it tests a translation prompt against a local Ollama model with no API key required.

To run it:
```
# From material/Session 7/getting-started/
npx promptfoo eval
npx promptfoo view
```

The `promptfooconfig.yaml` shows the core pattern: prompts → providers → test cases with assertions.

In [ ]:
# Display the promptfoo config from the getting-started example
with open("../getting-started/promptfooconfig.yaml") as f:
    print(f.read())

**When to use promptfoo vs. a Python eval loop:**
- Use **promptfoo** when you want to compare multiple models/prompts visually, or need a shareable report.
- Use a **Python eval loop** (below) when you need custom scoring logic, integration with your own data, or fine-grained control over the eval process.

Both patterns are complementary — promptfoo for exploration, Python loops for production eval pipelines.

---

### 2b — DIY Eval Loop

The core pattern for any eval pipeline:
1. Define test cases (input + expected output)
2. Run the model on each input
3. Score each output
4. Aggregate results

Let's build this for a sentiment analysis task — the same task we used in the prompting notebook.

In [ ]:
# 8 sentiment test cases: input text + expected label
test_cases = [
    {"input": "I absolutely loved the new restaurant! The food was amazing.", "expected": "Positive"},
    {"input": "The movie was a complete waste of time. Terrible plot.", "expected": "Negative"},
    {"input": "The package arrived on time. Nothing special.", "expected": "Neutral"},
    {"input": "Despite the wait, the food was delicious and staff were friendly.", "expected": "Positive"},
    {"input": "I'm so disappointed. The product arrived damaged and customer service ignored me.", "expected": "Negative"},
    {"input": "The hotel room was clean and functional. Met expectations.", "expected": "Neutral"},
    {"input": "Best coffee I've ever had! I'll definitely come back.", "expected": "Positive"},
    {"input": "Not the worst, but not great either. Average in every way.", "expected": "Neutral"},
]

print(f"Test set: {len(test_cases)} cases")
for i, tc in enumerate(test_cases, 1):
    print(f"  {i}. [{tc['expected']:8}] {tc['input'][:60]}...")

In [ ]:
SENTIMENT_PROMPT = """Classify the sentiment of the following text as exactly one of: Positive, Negative, or Neutral.
Respond with only the label — no explanation.

Text: {text}
Sentiment:"""

In [ ]:
def run_model(text):
    """Run the model on a single input and return the output."""
    prompt = SENTIMENT_PROMPT.format(text=text)
    return chat(prompt, temperature=0.0, max_tokens=10)

def exact_match(prediction, expected):
    """Check if the expected label appears in the prediction (case-insensitive)."""
    return expected.lower() in prediction.lower()

In [ ]:
import json

results = []
for case in test_cases:
    prediction = run_model(case["input"])
    correct = exact_match(prediction, case["expected"])
    results.append({
        "input": case["input"],
        "expected": case["expected"],
        "prediction": prediction.strip(),
        "correct": correct,
    })

accuracy = sum(r["correct"] for r in results) / len(results)
print(f"Accuracy: {accuracy:.1%}  ({sum(r['correct'] for r in results)}/{len(results)} correct)\n")

for r in results:
    icon = "\u2713" if r["correct"] else "\u2717"
    print(f"[{icon}] Expected: {r['expected']:8} | Got: {r['prediction']:12} | {r['input'][:55]}...")

**What to notice:**
- This is the fundamental eval loop pattern. Everything else is variations of: run → score → aggregate.
- Look at the **failures** — are they systematically wrong on neutral cases? Ambiguous reviews? This tells you where to improve the prompt.
- The prompt uses `temperature=0.0` for consistency — you want deterministic labels for evaluation.
- `exact_match` is a simple scorer. It works for classification. For open-ended tasks, you need something smarter — which is where LLM-as-judge comes in.

---

### 2c — LLM-as-Judge

For open-ended tasks (summarisation, explanation, QA), there's no single correct answer — `exact_match` doesn't work. **LLM-as-judge** uses a strong model to evaluate the output of another model against a rubric.

Pattern:
1. Define a rubric (scoring criteria)
2. Send the question, response, and rubric to a judge model
3. Ask for a score + justification in JSON format
4. Use `temperature=0` for the judge — you want consistent, deterministic scoring

In [ ]:
def llm_judge(question, response, rubric, judge_model="gpt-4o-mini"):
    """Use an LLM to score a response against a rubric. Returns (score, justification)."""
    prompt = f"""You are an expert evaluator. Score the following response using the rubric provided.

Question: {question}
Response: {response}
Rubric: {rubric}

Respond with ONLY a JSON object: {{"score": <int 1-5>, "justification": "<2 sentences>"}}"""
    
    raw = chat(prompt, temperature=0, max_tokens=200)
    
    # Parse JSON response
    try:
        # Handle cases where the model wraps JSON in markdown
        cleaned = raw.strip().strip("```json").strip("```").strip()
        data = json.loads(cleaned)
        return data["score"], data["justification"]
    except json.JSONDecodeError:
        return None, f"Parse error: {raw}"

In [ ]:
# Test the judge on a summarisation task
summarisation_rubric = """
5 - Captures all key points, concise, no hallucinations or omissions
4 - Captures most key points, minor omissions only
3 - Captures some key points, notable gaps or inaccuracies
2 - Mostly misses key points or contains significant errors
1 - Completely wrong, irrelevant, or incoherent
"""

source_article = (
    "Researchers at MIT have developed a new battery technology that charges in under 2 minutes "
    "and holds three times more energy than conventional lithium-ion batteries. The breakthrough "
    "uses a novel solid-state electrolyte and could enable electric vehicles to travel over 1,000 km "
    "on a single charge. Commercial production is expected within 5 years."
)

summaries_to_evaluate = [
    ("Good summary", "MIT researchers created a battery that charges in 2 minutes, holds 3x more energy than lithium-ion, enabling EVs to travel 1,000+ km per charge, with commercial availability in 5 years."),
    ("Partial summary", "MIT researchers made a new battery that charges fast and holds more energy."),
    ("Hallucinated summary", "Stanford researchers developed a solar-powered battery that can charge in 30 seconds and lasts a lifetime. It will be available next year for $99."),
]

question = f"Summarise this article in one sentence: {source_article}"

print(f"{'Summary type':<20} {'Score':>7} Justification")
print("-" * 90)
for name, summary in summaries_to_evaluate:
    score, justification = llm_judge(question, summary, summarisation_rubric)
    print(f"{name:<20} {str(score):>7}  {justification[:70]}...")

In [ ]:
# Demonstrate positional bias: judge prefers the response it sees first

good_summary = "MIT researchers created a battery that charges in 2 minutes and holds 3x more energy than lithium-ion, enabling EVs to travel over 1,000 km, with commercial production expected in 5 years."
poor_summary = "MIT researchers made a new battery that is better than current ones."

# Case A: good summary first
score_a_good, _ = llm_judge(
    "Compare and rate these two summaries of the same article.",
    f"Option A: {good_summary}\nOption B: {poor_summary}",
    "Rate Option A: which option is the better summary? Score 1-5."
)

# Case B: poor summary first (same content, swapped order)
score_b_poor, _ = llm_judge(
    "Compare and rate these two summaries of the same article.",
    f"Option A: {poor_summary}\nOption B: {good_summary}",
    "Rate Option A: which option is the better summary? Score 1-5."
)

print("Positional bias demo:")
print(f"  Good summary in Position A \u2192 Score for A: {score_a_good}")
print(f"  Poor summary in Position A \u2192 Score for A: {score_b_poor}")
print()
print("If scores differ, the judge is showing positional bias — rating 'Option A' higher regardless of content.")
print("Mitigation: swap order and average the two scores.")

**What to notice:**
- LLM-as-judge handles open-ended tasks that `exact_match` cannot.
- The hallucinated summary should score very low — the judge can detect factual errors.
- The positional bias demo shows a real limitation: judges sometimes prefer whatever comes first.
- **Mitigations**: run each pair twice (swapped), use multiple judge models, calibrate against human judgements.

---

### 2d — OpenAI Evals API (Optional Extension)

*(Requires OpenAI API key with Evals access — this section is optional)*

The OpenAI Evals API is the managed version of what we just built. You define the same pattern — data source + grading criteria — but OpenAI handles execution at scale.

The core graders mirror our DIY versions exactly:

| OpenAI grader | DIY equivalent |
|---|---|
| `string_check` | `exact_match()` |
| `label_model` | `llm_judge()` |
| `score_model` | `llm_judge()` with numeric output |

**`string_check` grader** (equivalent to `exact_match`):
```json
{
    "type": "string_check",
    "name": "Match sentiment label",
    "input": "{{ sample.output_text }}",
    "operation": "ilike",
    "reference": "{{ item.expected_label }}"
}
```

**`label_model` grader** (equivalent to `llm_judge` for classification):
```json
{
    "type": "label_model",
    "model": "gpt-4o-mini",
    "name": "Sentiment judge",
    "input": [
        {"role": "system", "content": "Classify the sentiment as Positive, Neutral, or Negative."},
        {"role": "user", "content": "Text: {{ item.input }}"}
    ],
    "labels": ["Positive", "Neutral", "Negative"],
    "passing_labels": ["Positive"]
}
```

**Key insight:** Knowing the DIY version makes you a better user of the platform. The managed API is an abstraction over exactly what you built above — but it scales to thousands of test cases, stores results, and enables regression testing across model versions.

Reference: [platform.openai.com/docs/guides/evals](https://platform.openai.com/docs/guides/evals)

---

## Summary — Three Tiers of LLM Evaluation

| Tier | What | Tools | When to use |
|---|---|---|---|
| **A — Classical metrics** | Surface/semantic overlap | BLEU, ROUGE, BERTScore, perplexity | Translation, summarisation benchmarks |
| **B — Eval tools** | Prompt testing at scale | Promptfoo, OpenAI Evals API | Compare prompts/models, regression testing |
| **C — Custom pipelines** | Task-specific scoring | DIY eval loop, LLM-as-judge | Your real project — AT2 and beyond |

**Which tier for your AT2 project?**
- If your task has a reference answer (translation, summarisation): start with ROUGE + BERTScore
- If your task is classification: use `exact_match` with your own test cases
- If your task is open-ended (QA, generation): use LLM-as-judge with a well-defined rubric
- Combine tiers — classical metrics for speed, LLM-as-judge for quality assessment

**Remember:** Benchmark scores tell you about general capability. Your own eval tells you about your task. The further your task is from standard benchmarks, the more important your own eval becomes.